In [1]:
import pandas as pd
import joblib, os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv('../data/raw/diabetes_012_health_indicators_BRFSS2015.csv')  # or chronic_sample.csv

features = ['HighBP', 'HighChol', 'BMI', 'Smoker', 'PhysActivity', 'GenHlth', 'Age', 'Sex']
target = 'Diabetes_012'

model_df = df[features + [target]].copy()
print(model_df.shape)
print("Missing:", model_df.isnull().sum().sum())
print("Duplicates:", model_df.duplicated().sum(), "(kept, not removed)")
print((model_df[target].value_counts(normalize=True).sort_index() * 100).round(2))

(253680, 9)
Missing: 0
Duplicates: 197586 (kept, not removed)
Diabetes_012
0.0    84.24
1.0     1.83
2.0    13.93
Name: proportion, dtype: float64


In [2]:
X = model_df.drop(columns=[target])
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train %:\n", (y_train.value_counts(normalize=True).sort_index() * 100).round(2))
print("Test %:\n", (y_test.value_counts(normalize=True).sort_index() * 100).round(2))

Train: (202944, 8) Test: (50736, 8)
Train %:
 Diabetes_012
0.0    84.24
1.0     1.83
2.0    13.93
Name: proportion, dtype: float64
Test %:
 Diabetes_012
0.0    84.24
1.0     1.83
2.0    13.93
Name: proportion, dtype: float64


In [3]:
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Test Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Test Accuracy: 83.01 %
              precision    recall  f1-score   support

         0.0       0.87      0.95      0.91     42741
         1.0       0.04      0.01      0.01       926
         2.0       0.41      0.22      0.29      7069

    accuracy                           0.83     50736
   macro avg       0.44      0.39      0.40     50736
weighted avg       0.79      0.83      0.80     50736

[[40549    87  2105]
 [  797     5   124]
 [ 5461    48  1560]]


In [4]:
model = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                               random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Test Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Test Accuracy: 61.75 %
              precision    recall  f1-score   support

         0.0       0.92      0.64      0.76     42741
         1.0       0.02      0.21      0.04       926
         2.0       0.28      0.52      0.37      7069

    accuracy                           0.62     50736
   macro avg       0.41      0.46      0.39     50736
weighted avg       0.82      0.62      0.69     50736

[[27454  6359  8928]
 [  336   198   392]
 [ 1974  1415  3680]]


In [5]:
y_binary = y.replace({0: 0, 1: 1, 2: 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, stratify=y_binary, random_state=42
)

model = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                               random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Test Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Test Accuracy: 71.57 %
              precision    recall  f1-score   support

         0.0       0.92      0.73      0.81     42741
         1.0       0.31      0.65      0.42      7995

    accuracy                           0.72     50736
   macro avg       0.61      0.69      0.62     50736
weighted avg       0.82      0.72      0.75     50736

[[31086 11655]
 [ 2771  5224]]


In [6]:
os.makedirs('../models', exist_ok=True)
joblib.dump(model, '../models/chronic_model.pkl')
joblib.dump(list(X_train.columns), '../models/chronic_features.pkl')
print("Saved. Feature order:", list(X_train.columns))

Saved. Feature order: ['HighBP', 'HighChol', 'BMI', 'Smoker', 'PhysActivity', 'GenHlth', 'Age', 'Sex']
